In [1]:
import pandas as pd
import chainladder as cl
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

auto_bi = cl.load_sample('friedland_auto_bi_insurer')


In [2]:
# Page 140, Exhibit I Sheet 1Column 1-2
auto_bi = cl.load_sample("friedland_auto_bi_insurer")
auto_bi["Reported Claims"].latest_diagonal

,2008
2000,"10,000,000"
2001,"8,000,000"
2002,"9,400,000"
2003,"15,600,000"
2004,"16,500,000"
2005,"18,500,000"
2006,"16,500,000"
2007,"14,000,000"
2008,"8,700,000"


In [3]:
# Page 140, Exhibit I Sheet 1 Column 3
auto_bi["Paid Claims"].latest_diagonal

,2008
2000,"9,500,000"
2001,"7,200,000"
2002,"7,600,000"
2003,"7,800,000"
2004,"11,200,000"
2005,"10,200,000"
2006,"6,000,000"
2007,"3,000,000"
2008,"750,000"


In [4]:
# Page 140, Exhibit I Sheet 1 Columns 4-5
# Direct age(in months): value representation per LOB
reported_pattern = {12: 4.0, 24: 2.9, 36: 1.8, 48: 1.4, 60: 1.2, 72: 1.1, 84: 1.03, 96: 1.02, 108: 1.005}
paid_pattern = {12: 90.0, 24: 15.0, 36: 5.0, 48: 2.5, 60: 1.75, 72: 1.35, 84: 1.25, 96: 1.15, 108: 1.05}

In [5]:
# Page 140, Exhibit I Sheet 1 Column 6
Reported_BI = cl.DevelopmentConstant(patterns=reported_pattern, style='cdf').fit_transform(auto_bi["Reported Claims"])
reported_ultimate = cl.Chainladder().fit(Reported_BI).ultimate_
reported_ultimate

,2261
2000,"10,050,000"
2001,"8,160,000"
2002,"9,682,000"
2003,"17,160,000"
2004,"19,800,000"
2005,"25,900,000"
2006,"29,700,000"
2007,"40,600,000"
2008,"34,800,000"


In [6]:
# Page 140, Exhibit I Sheet 1 Column 7
Paid_BI = cl.DevelopmentConstant(patterns=paid_pattern, style='cdf').fit_transform(auto_bi["Paid Claims"])
paid_ultimate = cl.Chainladder().fit(Paid_BI).ultimate_
paid_ultimate

,2261
2000,"9,975,000"
2001,"8,280,000"
2002,"9,500,000"
2003,"10,530,000"
2004,"19,600,000"
2005,"25,500,000"
2006,"30,000,000"
2007,"45,000,000"
2008,"67,500,000"


In [7]:
# Page 140, Exhibit I Sheet 1 Column 8
inital_selected_ultiamte_claims = (reported_ultimate + paid_ultimate)/2
inital_selected_ultiamte_claims

,2261
2000,"10,012,500"
2001,"8,220,000"
2002,"9,591,000"
2003,"13,845,000"
2004,"19,700,000"
2005,"25,700,000"
2006,"29,850,000"
2007,"42,800,000"
2008,"51,150,000"


In [8]:
# Page 140, Exhibit I Sheet 1 Column 9
auto_bi["Earned Premium"].latest_diagonal

,2008
2000,"24,000,000"
2001,"18,000,000"
2002,"19,000,000"
2003,"23,000,000"
2004,"32,000,000"
2005,"47,000,000"
2006,"50,000,000"
2007,"57,000,000"
2008,"62,000,000"


In [9]:
# Page 140, Exhibit I Sheet 1 Column 10
trend_factors = np.round(cl.Trend(trends=[0.145], dates=[("2008-12-31", "2000-01-01")]).fit(auto_bi["Earned Premium"]).trend_.latest_diagonal,3)
trend_factors

,2008
2000,2.9540
2001,2.5800
2002,2.2530
2003,1.9680
2004,1.7190
2005,1.5010
2006,1.3110
2007,1.1450
2008,1.0000


In [10]:
# Page 140, Exhibit I Sheet 1 Column 11
tort_factors = [0.670, 0.670, 0.670, 0.670, 0.750, 1.0, 1.0, 1.0, 1.0]
tort_factors

[0.67, 0.67, 0.67, 0.67, 0.75, 1.0, 1.0, 1.0, 1.0]

In [11]:
# Page 140, Exhibit I Sheet 1 Column 12
trended_adj_ultimate_claims = trend_factors * inital_selected_ultiamte_claims * np.array(tort_factors).reshape(1, 1, -1, 1)
trended_adj_ultimate_claims

,2008
2000,"19,816,540"
2001,"14,209,092"
2002,"14,477,710"
2003,"18,255,463"
2004,"25,398,225"
2005,"38,575,700"
2006,"39,133,350"
2007,"49,006,000"
2008,"51,150,000"


In [12]:
# Page 140, Exhibit I Sheet 1 Column 13
trended_adjusted_claim_ratio = np.round(trended_adj_ultimate_claims/auto_bi["Earned Premium"].latest_diagonal,2)
trended_adjusted_claim_ratio

,2008
2000,0.8300
2001,0.7900
2002,0.7600
2003,0.7900
2004,0.7900
2005,0.8200
2006,0.7800
2007,0.8600
2008,0.8200


In [13]:
# Page 140, Exhibit I Sheet 1 Item 14
print("Average 2000 to 2005:", np.round(trended_adjusted_claim_ratio.iloc[:, :, 0:6, :].mean(),3))
loss_ratios_00_05 = trended_adjusted_claim_ratio.iloc[:, :, 0:6, :].values.flatten()
print("Average 2000 to 2005 Ex High Ex Low:", np.round((loss_ratios_00_05.sum() - loss_ratios_00_05.max() - loss_ratios_00_05.min()) / (len(loss_ratios_00_05) - 2), 3))
print("Average 2001 to 2006:", np.round(trended_adjusted_claim_ratio.iloc[:, :, 1:7, :].mean(),3))
loss_ratios_01_06 = trended_adjusted_claim_ratio.iloc[:, :, 1:7, :].values.flatten()
print("Average 2000 to 2005 Ex High Ex Low:", np.round((loss_ratios_01_06.sum() - loss_ratios_01_06.max() - loss_ratios_01_06.min()) / (len(loss_ratios_01_06) - 2), 3)) # small rounding error due to python's 64 bit float storage

Average 2000 to 2005: 0.797
Average 2000 to 2005 Ex High Ex Low: 0.798
Average 2001 to 2006: 0.788
Average 2000 to 2005 Ex High Ex Low: 0.787


In [14]:
# Page 140, Exhibit I Item 15
selected_claim_ratio = 0.80
selected_claim_ratio

0.8

In [15]:
# Page 140, Exhibit I Item 16
exlected_claims_2008 = auto_bi["Earned Premium"].loc[:, :, "2008", :].latest_diagonal * selected_claim_ratio
exlected_claims_2008


np.float64(49600000.0)

In [16]:
# Page 140, Exhibit I Item 17
print("Total Unpaid Claims:", exlected_claims_2008 - auto_bi["Paid Claims"].loc[:, :, "2008", :].latest_diagonal)
print("Total IBNR:", exlected_claims_2008 - auto_bi["Reported Claims"].loc[:, :, "2008", :].latest_diagonal)

Total Unpaid Claims: 48850000.0
Total IBNR: 40900000.0


In [ ]:
# Page 141, Exhibit I Sheet 2

In [ ]:
# Page 142, Exhibit II Sheet 1, columns 1-2
friedland_us_industry_auto = cl.load_sample("friedland_us_industry_auto")
friedland_us_industry_auto["Reported Claims"].latest_diagonal

,2007
1998,"47,742,304"
1999,"51,185,767"
2000,"54,837,929"
2001,"56,299,562"
2002,"58,592,712"
2003,"57,565,344"
2004,"56,976,657"
2005,"56,786,410"
2006,"54,641,339"
2007,"48,853,563"


In [ ]:
# Page 142, Exhibit II Sheet 1, columns 3
friedland_us_industry_auto["Paid Claims"].latest_diagonal

,2007
1998,"47,644,187"
1999,"51,000,534"
2000,"54,533,225"
2001,"55,878,421"
2002,"57,807,215"
2003,"55,930,654"
2004,"53,774,672"
2005,"50,644,994"
2006,"43,606,497"
2007,"27,229,969"


In [ ]:
# Page 142, Exhibit II Sheet 1, columns 4-5
# Direct age(in months): value representation per LOB
reported_pattern = {12: 1.292, 24: 1.110, 36: 1.051, 48: 1.023, 60: 1.011, 72: 1.006, 84: 1.003, 96: 1.001, 108: 1.000, 120: 1.000}
paid_pattern = {12: 2.390, 24: 1.404, 36: 1.184, 48: 1.085, 60: 1.040, 72: 1.020, 84: 1.011, 96: 1.006, 108: 1.004, 120: 1.002}